In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

# TOP 5 risk locations based on NWS code so whole netwerkschakels (not distinguishing left and right)

In [2]:
from pathlib import Path
import warnings
import re
import pandas as pd
import geopandas as gpd

# Load input files
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")
gdf_area = gpd.read_file(area_gpkg)
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Losses_Analysis\aggregated_loss_per_nws_code.gpkg")
gdf_src = gpd.read_file(input_path)

# Overlay function
def add_area_overlay(gdf: gpd.GeoDataFrame, area_path: Path, target_crs: str, key_col: str) -> gpd.GeoDataFrame:
    import fiona
    if not area_path.exists():
        warnings.warn(f"Area file not found: {area_path}. Skipping area overlay.")
        return gdf.copy()
    layers = fiona.listlayers(str(area_path))
    layer = layers[0]
    area = gpd.read_file(area_path, layer=layer)
    if area.crs is None:
        area.set_crs(target_crs, inplace=True)
    if area.crs.to_string() != target_crs:
        area = area.to_crs(target_crs)
    candidate_labels = ["name", "id"]
    label_col = next((c for c in candidate_labels if c in area.columns), None)
    new_col = area_path.stem
    left = gdf[[key_col, "geometry"]].copy()
    right_cols = ["geometry"] + ([label_col] if label_col else [])
    sj = gpd.sjoin(left, area[right_cols], how="left", predicate="intersects")
    has_overlap = (
        sj.groupby(key_col)["index_right"]
          .apply(lambda s: s.notna().any())
          .rename(new_col)
          .reset_index()
    )
    out = gdf.merge(has_overlap, on=key_col, how="left")
    out[new_col] = out[new_col].fillna(False)
    if label_col:
        labels = (
            sj.groupby(key_col)[label_col]
              .apply(lambda s: ";".join(sorted({str(v) for v in s.dropna()})) if s.notna().any() else None)
              .rename(f"{new_col}_{label_col}")
              .reset_index()
        )
        out = out.merge(labels, on=key_col, how="left")
    return out

# Split Areas_name into individual entries
def split_areas(val):
    if pd.isna(val):
        return []
    parts = [p.strip() for p in re.split(r"[;,\u061B]", str(val))]
    return [p for p in parts if p]

# Apply overlay
TARGET_CRS = "EPSG:28992"
merged = add_area_overlay(gdf_src, area_gpkg, TARGET_CRS, key_col="NWSCODE")

# Ensure required columns
if "Areas_name" not in merged.columns:
    raise KeyError("Column 'Areas_name' not found after overlay. Make sure Areas.gpkg has a 'name' field.")
if "VHLH_AL_E_WR" not in merged.columns:
    raise KeyError("Column 'VHLH_AL_E_WR' not found in losses input.")

# Prepare, explode per Area
merged["VHLH_AL_E_WR"] = pd.to_numeric(merged["VHLH_AL_E_WR"], errors="coerce")
gdf_loss_exp = (
    merged.assign(Area=merged["Areas_name"].apply(split_areas))
          .explode("Area", ignore_index=True)
)
gdf_loss_exp = gdf_loss_exp[gdf_loss_exp["Area"].notna() & (gdf_loss_exp["Area"] != "")]

# Custom ranking: assign max rank to missing or zero values per Area
def custom_rank(group):
    series = group["VHLH_AL_E_WR"]
    ranks = series.rank(ascending=False, method="min")
    ranks[series.isna() | (series == 0)] = len(series)
    return ranks

gdf_loss_exp["rk_VHLH_AL_E_WR"] = gdf_loss_exp.groupby("Area", group_keys=False).apply(custom_rank)

# Sort for readability
gdf_loss_ranked = (
    gdf_loss_exp
      .sort_values(["Area", "rk_VHLH_AL_E_WR", "VHLH_AL_E_WR"], ascending=[True, True, False], kind="mergesort")
      .reset_index(drop=True)
)

# Output paths
base_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
base_dir.mkdir(parents=True, exist_ok=True)
gpkg_path = base_dir / "Simple_Losses_ranking.gpkg"
shp_path  = base_dir / "Simple_Losses_ranking.shp"

# Save GPKG
gdf_loss_ranked.to_file(gpkg_path, layer="ranked_rows", driver="GPKG")

# Save SHP (shorten field names)
gdf_loss_ranked_shp = gdf_loss_ranked.copy()
gdf_loss_ranked_shp["rk_VHLH_AL_E_WR"] = gdf_loss_ranked_shp["rk_VHLH_AL_E_WR"].fillna(-1).astype("int32")
gdf_loss_ranked_shp = gdf_loss_ranked_shp.rename(columns={"rk_VHLH_AL_E_WR": "rk_VHLH"})

# Clean old shapefile sidecars
def delete_shapefile(path: Path):
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg", ".shp.xml", ".qpj"]:
        p = path.with_suffix(ext)
        if p.exists():
            p.unlink()

delete_shapefile(shp_path)
gdf_loss_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")

print(f"Wrote:\n- {gpkg_path}\n- {shp_path}")

CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: table gpkg_extensions already exists'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: table gpkg_extensions already exists'
C:\Users\meije_le\AppData\Local\Temp\ipykernel_25448\2931410699.py:113: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_loss_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")


Wrote:
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking.gpkg
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking.shp


In [3]:
#fixing the damages:


import geopandas as gpd
from pathlib import Path

# Read the input file
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area_edited_ver04.gpkg")
gdf_src = gpd.read_file(input_path)

# Dissolve by 'NET' column with custom aggregation
gdf_dissolved = gdf_src.dissolve(
    by="NET",
    aggfunc={
        "total_length": "sum",
        "flooded_length": "sum",
        "bridge_length_sum": "sum",
        "tunnel_length_sum": "sum",
        "total_damage": "sum",
        "Areas_name": lambda x: ', '.join(x.unique())  # Combine unique values
    }
)

# Reset index if needed
gdf_dissolved = gdf_dissolved.reset_index()
#gdf_dissolved = gdf_dissolved["NET"].rename(columns={"NET": "NWSCODE"}, inplace=True)
gdf_dissolved.rename(columns={"NET": "NWSCODE"}, inplace=True)

# Save to a new file
output_path = input_path.parent / "Damages_Aggregated_by_Schakels.gpkg"
gdf_dissolved.to_file(output_path, driver="GPKG")

print(f"Dissolved file saved to: {output_path}")


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Damages_Aggregated_by_Schakels')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Damages_Aggregated_by_Schakels')) failed: unable to open database file"


Dissolved file saved to: P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damages_Aggregated_by_Schakels.gpkg


In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import re

# Input
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damages_Aggregated_by_Schakels.gpkg")
gdf_src = gpd.read_file(input_path)

# Split Areas_name into individual entries
def split_areas(val):
    if pd.isna(val):
        return []
    parts = [p.strip() for p in re.split(r"[;,\u061B]", str(val))]  # split on ; , and Arabic semicolon
    #return [p for p in parts if p]
    # Use a dictionary to preserve order while getting unique values
    return list(dict.fromkeys([p for p in parts if p]))

if "Areas_name" not in gdf_src.columns:
    raise KeyError("Column 'Areas_name' not found.")

gdf_exp = (
    gdf_src.assign(Area=gdf_src["Areas_name"].apply(split_areas))
           .explode("Area", ignore_index=True)
)
gdf_exp = gdf_exp[gdf_exp["Area"].notna() & (gdf_exp["Area"] != "")]

# Calculate metrics
gdf_exp["dam_per_m"] = (
    (gdf_exp["total_damage"] / gdf_exp["total_length"])
    .where(gdf_exp["total_length"].notna() & (gdf_exp["total_length"] != 0))
)

gdf_exp["fraction_flooded"] = (
    (gdf_exp["flooded_length"] / gdf_exp["total_length"])
    .where(gdf_exp["total_length"].notna() & (gdf_exp["total_length"] != 0))
)

# Custom ranking: assign max rank to missing or zero values
def custom_rank(series):
    ranks = series.rank(ascending=False, method="min")
    ranks[series.isna() | (series == 0)] = len(series)
    return ranks

gdf_exp["rank_frfl"] = gdf_exp.groupby("Area")["fraction_flooded"].transform(custom_rank)
gdf_exp["rk_d_m"] = gdf_exp.groupby("Area")["dam_per_m"].transform(custom_rank)

# Prepare ranked output
required = ["Area",'Areas_name', "rank_frfl", "rk_d_m", "dam_per_m", "NWSCODE", "total_damage", "fraction_flooded",
            "total_length", "flooded_length", "tunnel_length_sum", "bridge_length_sum", "geometry"]
missing = [c for c in required if c not in gdf_exp.columns]
if missing:
    raise KeyError(f"Missing in gdf_exp: {missing}")

gdf_ranked = (
    gdf_exp.loc[:, required]
            .sort_values(["Area", "rank_frfl", "rk_d_m", "dam_per_m", "fraction_flooded"],
                         ascending=[True, True, True, False, False], kind="mergesort")
            .reset_index(drop=True)
)

# Output paths
base_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
base_dir.mkdir(parents=True, exist_ok=True)
gpkg_path = base_dir / "Damage_risk_ranking_schakels.gpkg"
shp_path  = base_dir / "Damage_risk_ranking_schakels.shp"

# Save GPKG
gdf_ranked.to_file(gpkg_path, layer="ranked_rows", driver="GPKG")

# Save SHP (shorten field names)
gdf_ranked_shp = gdf_ranked.rename(columns={
    "total_damage": "tot_dam",
    "fraction_flooded": "frac_fld",
    "rk_d_m": "rk_d_m",
    "rank_frfl": "rk_frfl"
}).copy()

# Convert ranks to integer for SHP
gdf_ranked_shp["rk_d_m"] = gdf_ranked_shp["rk_d_m"].astype("Int64")
gdf_ranked_shp["rk_frfl"] = gdf_ranked_shp["rk_frfl"].astype("Int64")

gdf_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")

print(f"Wrote:\n- {gpkg_path}\n- {shp_path}")

In [ ]:
gdf

,NWSCODE,Area,Areas_name,rk_VHLH,rk_d_m,flooded_length,flooded_rank,tunnel_length_sum,inv_VHLH,inv_Damage,inv_Tunnel,Total,Final_rank,geometry
0,004-0050,ARK-NZK,ARK-NZK,1,7.0,884.012273,7.0,2830.651392,62,56.0,70.0,60.6,1,"MULTILINESTRING ((97995.869 463841.288, 98031...."
1,028-0010,ARK-NZK,ARK-NZK;Vallei en Veluwe,8,2.0,11200.901425,1.0,2015.601084,55,61.0,67.0,60.4,2,"MULTILINESTRING ((157179.527 463506.622, 15717..."
2,028-0010,ARK-NZK,ARK-NZK;Vallei en Veluwe,8,3.0,11200.901425,1.0,2015.601084,55,60.0,67.0,59.9,3,"MULTILINESTRING ((157179.527 463506.622, 15717..."
3,010-0030,ARK-NZK,ARK-NZK,7,8.0,292.473925,21.0,144.817419,56,55.0,57.0,55.7,4,"MULTILINESTRING ((117775.052 484197.926, 11777..."
4,011-0010,ARK-NZK,ARK-NZK,9,15.0,604.085783,13.0,1330.800373,54,48.0,61.0,52.4,5,"MULTILINESTRING ((110267.315 455696.352, 11025..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,006-0080,Zuiderzeeland,Zuiderzeeland,8,8.0,0.000000,5.0,0.000000,1,22.0,1.0,11.5,6,"MULTILINESTRING ((180851.635 527114.102, 18085..."
400,027-0090,Zuiderzeeland,ARK-NZK;Vallei en Veluwe;Zuiderzeeland,1,14.0,1696.569532,1.0,0.000000,8,16.0,1.0,10.6,10,"MULTILINESTRING ((151123.623 480575.452, 15111..."
401,006-1010,Zuiderzeeland,Friesland;Zuiderzeeland,8,11.0,0.000000,5.0,0.000000,1,19.0,1.0,10.0,11,"MULTILINESTRING ((181086.289 547819.638, 18108..."
402,006-0020,Zuiderzeeland,ARK-NZK;Zuiderzeeland,3,22.0,695.814459,3.0,0.000000,6,8.0,1.0,6.0,12,"MULTILINESTRING ((144785.005 484995.896, 14477..."


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Load losses ranking (country-wide)
losses_rank_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking.shp")
losses_rank = gpd.read_file(losses_rank_path, driver="ESRI Shapefile")
print(losses_rank.columns)

# Paths for (country-wide merged) damages
root_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
gpkg_path = root_dir / "Damage_risk_ranking_schakels.gpkg"
damages_rank = gpd.read_file(gpkg_path, driver="GPKG")
print(damages_rank.columns)

Index(['NWSCODE', 'Length_sch', 'NWSNAAM', 'VHLH_AL_E_', 'VHLH_L1_E_',
       'VHLH_L2_E_', 'VHLH_L3_E_', 'F_EV2_ma', 'F_EV1_me', 'AL_E_WR',
       'getroffen_', 'getroffe_1', 'vracht_op_', 'personen_o', 'VOT_L1_gem',
       'VOT_L2L3_g', 'VOT_total', 'Areas', 'Areas_name', 'Area', 'rk_VHLH',
       'geometry'],
      dtype='object')
Index(['Area', 'Areas_name', 'rank_frfl', 'rk_d_m', 'dam_per_m', 'NWSCODE',
       'total_damage', 'fraction_flooded', 'total_length', 'flooded_length',
       'tunnel_length_sum', 'bridge_length_sum', 'geometry'],
      dtype='object')


In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Load losses ranking (country-wide)
losses_rank_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking.shp")
losses_rank = gpd.read_file(losses_rank_path, driver="ESRI Shapefile")

# Paths for (country-wide merged) damages
root_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
gpkg_path = root_dir / "Damage_risk_ranking_schakels.gpkg"

# Read damages
damages_rank = gpd.read_file(gpkg_path, driver="GPKG")

# Optional: damage per meter
# damages_rank["dam_per_m"] = (
#     (damages_rank["total_damage"] / damages_rank["total_length"])
#     .where(damages_rank["total_length"].notna() & (damages_rank["total_length"] != 0))
# )

# Merge VHLH and damages (country-wide), keep Area for groupwise ranking
vhlh_df = losses_rank[['NWSCODE', 'NWSNAAM', 'rk_VHLH', 'F_EV2_ma', 'Area']].copy()
damage_df = damages_rank[['NWSCODE', 'rk_d_m', 'Area', 'Areas_name', 'flooded_length', 'tunnel_length_sum', 'geometry']].copy()

df = pd.merge(damage_df, vhlh_df, on=['NWSCODE', 'Area'], how='inner')

# Convert back to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=damages_rank.crs)

# ✅ Invert ranks for VHLH and Damage per Area
max_vhlh_by_area = gdf.groupby('Area')['rk_VHLH'].transform('max')
max_damage_by_area = gdf.groupby('Area')['rk_d_m'].transform('max')
gdf['inv_VHLH'] = (max_vhlh_by_area + 1) - gdf['rk_VHLH']
gdf['inv_Damage'] = (max_damage_by_area + 1) - gdf['rk_d_m']

# ✅ Tunnel ranking: assign max rank to missing or zero values per Area
def custom_rank(series):
    ranks = series.rank(ascending=False, method="min")
    ranks[series.isna() | (series == 0)] = len(series)
    return ranks

gdf['tunnel_rank'] = gdf.groupby('Area')['tunnel_length_sum'].transform(custom_rank)
max_tunnel_by_area = gdf.groupby('Area')['tunnel_rank'].transform('max')
gdf['inv_Tunnel'] = (max_tunnel_by_area + 1) - gdf['tunnel_rank']

# ✅ Rank flooded_length within Area
gdf['flooded_rank'] = gdf.groupby('Area')['flooded_length'].rank(method='dense', ascending=False)

# ✅ Compute weighted score (Damage, VHLH, Tunnel)
weights = {'Damage': 0.5, 'VHLH': 0.3, 'Tunnels': 0.2}
gdf['Total'] = (
    gdf['inv_VHLH'] * weights['VHLH'] +
    gdf['inv_Damage'] * weights['Damage'] +
    gdf['inv_Tunnel'] * weights['Tunnels']
)

gdf = gdf.sort_values(['Area', 'Total'], ascending=[True, False]).reset_index(drop=True)

def competition_rank(group):
    return group['Total'].rank(method='min', ascending=False).astype(int)

gdf['Final_rank'] = gdf.groupby('Area', group_keys=False).apply(competition_rank)

# ✅ Keep flooded_length and flooded_rank in final output
columns_to_keep = [
    'NWSCODE', 'NWSNAAM', 'Area', 'Areas_name', 'F_EV2_ma', 'rk_VHLH', 'rk_d_m',
    'flooded_length', 'flooded_rank', 'tunnel_length_sum',
    'inv_VHLH', 'inv_Damage', 'inv_Tunnel', 'Total', 'Final_rank', 'geometry'
]
gdf = gdf[columns_to_keep]

# Save output
out_path = root_dir / "Combined_Risk_Ranking_country_scale_NWSCODE_wholeschakel_ver05.gpkg"
gdf.to_file(out_path, driver="GPKG")

print(f"Saved ranking to {out_path}")

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking_country_scale_NWSCODE_wholeschakel_ver05')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking_country_scale_NWSCODE_wholeschakel_ver05')) failed: unable to open database file"


Saved ranking to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Combined_Risk_Ranking_country_scale_NWSCODE_wholeschakel_ver05.gpkg


In [ ]:
print(losses_rank['NWSCODE'].nunique(), len(losses_rank))
print(damages_rank['NWSCODE'].nunique(), len(damages_rank))

244 296
244 296


In [ ]:
# import geopandas as gpd
# import pandas as pd
# from pathlib import Path

# # Regions list
# region_list = [
#     "Groningen en NO-Drenthe", "Noord-Westelijke Delta", "Overijsselse Vecht",
#     "Limburg", "Vallei en Veluwe", "Achterhoek", "Brabantse Delta",
#     "Friesland", "ARK-NZK", "Noord-Brabant Oost", "Rivierenland",
#     "Scheldestromen", "Zuiderzeeland"
# ]

# # Load losses ranking (country-wide)
# losses_rank_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking.shp")
# losses_rank = gpd.read_file(losses_rank_path, driver="ESRI Shapefile")

# for region in region_list:
#     print(f"Processing region: {region}")
    
#     # Paths for region-specific damages
#     root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
#     gpkg_path = root_dir / "ranked_Aggregated_schakels.gpkg"
    
#     # Read damages for this region
#     damages_rank = gpd.read_file(gpkg_path, driver="GPKG")
    
#     # Filter losses for this region
#     losses_rank_region = losses_rank[losses_rank["Area"] == region]
    
#     # Merge VHLH and damages
#     vhlh_df = losses_rank_region[['NWSCODE', 'rk_VHLH', 'Area', 'Areas_name']].copy()
#     damage_df = damages_rank[['NWSCODE', 'rk_d_m', 'flooded_length', 'tunnel_length_sum', 'geometry']].copy()
    
#     df = pd.merge(damage_df, vhlh_df, left_on='NWSCODE', right_on='NWSCODE', how='inner')
    
#     # Fill missing tunnel lengths with 0
#     df['tunnel_length_sum'] = df['tunnel_length_sum'].fillna(0)
    
#     # Convert back to GeoDataFrame
#     gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=damages_rank.crs)
    
#     # ✅ Invert ranks for VHLH and Damage
#     max_vhlh = gdf['rk_VHLH'].max()
#     max_damage = gdf['rk_d_m'].max()
#     gdf['inv_VHLH'] = (max_vhlh + 1) - gdf['rk_VHLH']
#     gdf['inv_Damage'] = (max_damage + 1) - gdf['rk_d_m']
    
#     # ✅ Rank tunnels within region and invert
#     gdf['tunnel_rank'] = gdf['tunnel_length_sum'].rank(method='dense', ascending=False)
#     max_tunnel = gdf['tunnel_rank'].max()
#     gdf['inv_Tunnel'] = (max_tunnel + 1) - gdf['tunnel_rank']
    
#     # ✅ Rank flooded_length (no weighting yet)
#     gdf['flooded_rank'] = gdf['flooded_length'].rank(method='dense', ascending=False)
    
#     # ✅ Compute weighted score (Damage, VHLH, Tunnel)
#     weights = {'Damage': 0.5, 'VHLH': 0.3, 'Tunnels': 0.2}
#     gdf['Total'] = (gdf['inv_VHLH'] * weights['VHLH'] +
#                     gdf['inv_Damage'] * weights['Damage'] +
#                     gdf['inv_Tunnel'] * weights['Tunnels'])
    
#     # Sort and assign final rank
#     gdf = gdf.sort_values('Total', ascending=False)
#     gdf['Final_rank'] = range(1, len(gdf)+1)
    
#     # ✅ Keep flooded_length and flooded_rank in final output
#     columns_to_keep = ['NWSCODE', 'rk_VHLH', 'rk_d_m', 'flooded_length', 'flooded_rank',
#                        'tunnel_length_sum', 'inv_VHLH', 'inv_Damage', 'inv_Tunnel',
#                        'Total', 'Final_rank', 'geometry']
#     gdf = gdf[columns_to_keep]
    
#     # Save output for this region
#     out_path = root_dir / "Combined_Risk_Ranking.gpkg"
#     gdf.to_file(out_path, driver="GPKG")
    
#     print(f"Saved ranking for {region} to {out_path}")

Processing region: Groningen en NO-Drenthe


KeyError: "['NWSCODE'] not in index"